# Test-set evaluation — all four encoder conditions

Runs the held-out test set once, through the same model classes, tokenisers and
decoding paths used during training. No training happens here: checkpoints are
loaded and a single forward pass is made over `data/test.jsonl`.

Writes `test_metrics.json` and `test_predictions.csv` into each run directory,
in the same column format as the existing `dev_predictions.csv` (including
`p_pred`), so the downstream table and plotting code works unchanged.

Model classes, tokenisers, `max_length`, label filtering, CORN decoding and the
ModernBERT `prediction_step` override are all taken from the two PROBABILITIES
notebooks. Checkpoints resolve to `best_model_checkpoint` from
`trainer_state.json` automatically; set `ckpt=` on a condition to override.

In [1]:
import os, json, csv, gc, glob
from dataclasses import dataclass

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer, AutoModel, Trainer, TrainingArguments, DataCollatorWithPadding,
)
from transformers.modeling_outputs import SequenceClassifierOutput
from safetensors.torch import load_file

from coral_pytorch.losses import corn_loss
from coral_pytorch.dataset import corn_label_from_logits
from metrics import evaluate_predictions

LEVELS = ["A2", "A2+", "B1", "B1+", "B2", "B2+", "C1", "C1+"]
LABEL2ID = {lvl: i for i, lvl in enumerate(LEVELS)}
NUM_CLASSES = len(LEVELS)


def corn_probas(logits):
    """(N, K-1) CORN logits -> (N, K) class distribution, rows sum to 1."""
    cond = torch.sigmoid(logits)
    cum = torch.cumprod(cond, dim=1)
    p = torch.zeros(logits.size(0), logits.size(1) + 1)
    p[:, 0] = 1.0 - cum[:, 0]
    p[:, 1:-1] = cum[:, :-1] - cum[:, 1:]
    p[:, -1] = cum[:, -1]
    return p.clamp_min(0.0)

In [2]:
@dataclass
class Cond:
    name: str
    family: str          # "t5gemma" | "modernbert"
    mode: str            # "corn" | "nominal"
    model_id: str
    output_dir: str
    ckpt: str = ""       # blank -> auto-resolve best_model_checkpoint
    max_length: int = 1024
    batch_size: int = 8
    bf16: bool = True
    custom_prediction_step: bool = False   # ModernBERT notebook overrides it


TEST_PATH = "data/test.jsonl"

CONDITIONS = [
    Cond("T5Gemma CORN",       "t5gemma",    "corn",
         "google/t5gemma-2-4b-4b", "runs/t5gemma2_corn",  ckpt="checkpoint-1900"),
    Cond("T5Gemma nominal",    "t5gemma",    "nominal",
         "google/t5gemma-2-4b-4b", "runs/t5gemma2_nominal", ckpt="checkpoint-1900"),
    Cond("ModernBERT CORN",    "modernbert", "corn",
         "answerdotai/ModernBERT-large", "runs/modernbert_corn",
         custom_prediction_step=True),
    Cond("ModernBERT nominal", "modernbert", "nominal",
         "answerdotai/ModernBERT-large", "runs/modernbert_nominal",
         custom_prediction_step=True),
]


class ModernBERTOrdinalClassifier(nn.Module):
    # Verbatim from PROBABILITIES_modernbert_cefr.ipynb
    def __init__(self, model_id, num_classes=8, mode="corn", dropout=0.1, dtype=None):
        super().__init__()
        assert mode in {"corn", "nominal"}
        self.mode = mode
        self.encoder = AutoModel.from_pretrained(model_id, torch_dtype=dtype)
        hidden = self.encoder.config.hidden_size
        out_dim = num_classes - 1 if mode == "corn" else num_classes
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Linear(hidden, out_dim)
        self.head.to(torch.float32)

    def forward(self, input_ids=None, attention_mask=None, **kwargs):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        h = out.last_hidden_state
        if attention_mask is not None:
            mask = attention_mask.unsqueeze(-1).to(h.dtype)
            pooled = (h * mask).sum(1) / mask.sum(1).clamp(min=1)
        else:
            pooled = h.mean(1)
        logits = self.head(self.dropout(pooled).to(torch.float32))
        return SequenceClassifierOutput(logits=logits)


def build_model(c):
    dtype = torch.bfloat16 if c.bf16 else torch.float32
    if c.family == "t5gemma":
        from model import T5Gemma2OrdinalClassifier
        return T5Gemma2OrdinalClassifier(
            c.model_id, num_classes=NUM_CLASSES, mode=c.mode, dtype=dtype)
    if c.family == "modernbert":
        return ModernBERTOrdinalClassifier(
            c.model_id, num_classes=NUM_CLASSES, mode=c.mode, dtype=dtype)
    raise ValueError(c.family)


def resolve_ckpt(c):
    # best_model_checkpoint from trainer_state.json, unless c.ckpt is set
    if c.ckpt:
        return os.path.join(c.output_dir, c.ckpt)
    states = sorted(glob.glob(os.path.join(c.output_dir, "checkpoint-*",
                                           "trainer_state.json")))
    if not states:
        raise FileNotFoundError(f"no checkpoints under {c.output_dir}")
    best = json.load(open(states[-1])).get("best_model_checkpoint")
    if not best:
        raise ValueError(f"{c.output_dir}: no best_model_checkpoint recorded; "
                         "set ckpt= explicitly")
    path = os.path.join(c.output_dir, os.path.basename(best))
    if not os.path.isdir(path):
        raise FileNotFoundError(f"best checkpoint {path} was pruned; "
                                "set ckpt= to a surviving one")
    return path

In [3]:
def read_rows(path):
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                yield json.loads(line)


class CEFRDataset(Dataset):
    # Same label filtering and truncation as training
    def __init__(self, path, tokenizer, max_length,
                 text_col="text", label_col="label"):
        self.ex, skipped = [], 0
        for row in read_rows(path):
            lab = str(row[label_col]).strip().upper()
            if lab not in LABEL2ID:
                skipped += 1
                continue
            self.ex.append((str(row[text_col]), LABEL2ID[lab]))
        print(f"{path}: kept {len(self.ex)}, skipped {skipped} out-of-scope")
        self.tok, self.max_length = tokenizer, max_length

    def __len__(self):
        return len(self.ex)

    def __getitem__(self, i):
        text, label = self.ex[i]
        enc = self.tok(text, truncation=True, max_length=self.max_length)
        enc["labels"] = label
        return enc


def _loss(logits, labels, mode):
    return (corn_loss(logits, labels.long(), NUM_CLASSES) if mode == "corn"
            else torch.nn.functional.cross_entropy(logits, labels.long()))


class OrdinalTrainer(Trainer):
    def __init__(self, *a, mode="corn", **k):
        super().__init__(*a, **k)
        self.mode = mode

    def compute_loss(self, model, inputs, return_outputs=False, **kw):
        labels = inputs.pop("labels")
        out = model(**inputs)
        return ((_loss(out.logits, labels, self.mode), out) if return_outputs
                else _loss(out.logits, labels, self.mode))


class OrdinalTrainerCustomStep(OrdinalTrainer):
    # ModernBERT path: prediction_step returns (loss, logits, labels)
    def prediction_step(self, model, inputs, prediction_loss_only, ignore_keys=None):
        inputs = self._prepare_inputs(inputs)
        labels = inputs.pop("labels")
        with torch.no_grad():
            out = model(**inputs)
            logits = out.logits
            loss = _loss(logits, labels, self.mode)
        if prediction_loss_only:
            return (loss, None, None)
        return (loss, logits, labels)


def as_logits(preds):
    if isinstance(preds, tuple):
        preds = preds[0]
    return torch.as_tensor(np.asarray(preds, dtype=np.float32))


def decode(logits, mode):
    return (corn_label_from_logits(logits).numpy() if mode == "corn"
            else logits.argmax(-1).numpy())


def make_compute_metrics(mode):
    def fn(eval_pred):
        logits = as_logits(eval_pred.predictions)
        return evaluate_predictions(eval_pred.label_ids, decode(logits, mode))
    return fn

In [4]:
def run_test(c, overwrite=False):
    out_json = os.path.join(c.output_dir, "test_metrics.json")
    if os.path.exists(out_json) and not overwrite:
        print(f"SKIP {c.name}: test_metrics.json already exists")
        return

    ckpt = resolve_ckpt(c)
    print(f"=== {c.name} | {ckpt} ===")

    tokenizer = AutoTokenizer.from_pretrained(c.model_id)
    test_ds = CEFRDataset(TEST_PATH, tokenizer, c.max_length)

    model = build_model(c)
    sd_path = os.path.join(ckpt, "model.safetensors")
    sd = (load_file(sd_path) if os.path.exists(sd_path)
          else torch.load(os.path.join(ckpt, "pytorch_model.bin"), map_location="cpu"))
    missing, unexpected = model.load_state_dict(sd, strict=False)
    print(f"missing {len(missing)} | unexpected {len(unexpected)}")
    if missing:
        print("  e.g.", missing[:5])
    assert not any("head" in k for k in missing), "head weights failed to load"

    args = TrainingArguments(output_dir="tmp_eval",
                             per_device_eval_batch_size=c.batch_size,
                             bf16=c.bf16, report_to=[], do_train=False)
    TrainerCls = (OrdinalTrainerCustomStep if c.custom_prediction_step
                  else OrdinalTrainer)
    trainer = TrainerCls(model=model, args=args, mode=c.mode,
                         processing_class=tokenizer,
                         data_collator=DataCollatorWithPadding(tokenizer),
                         compute_metrics=make_compute_metrics(c.mode))

    out = trainer.predict(test_ds, metric_key_prefix="test")
    logits = as_logits(out.predictions)
    y_pred, y_true = decode(logits, c.mode), out.label_ids
    probs = (corn_probas(logits) if c.mode == "corn"
             else torch.softmax(logits, dim=-1)).numpy()

    print(f"logits {tuple(logits.shape)}  rows sum to 1: "
          f"{np.allclose(probs.sum(1), 1.0, atol=1e-4)}")
    print("test:", json.dumps({k: round(v, 4) for k, v in out.metrics.items()
                               if isinstance(v, float)}, indent=2))

    os.makedirs(c.output_dir, exist_ok=True)
    with open(out_json, "w") as f:
        json.dump(out.metrics, f, indent=2)
    with open(os.path.join(c.output_dir, "test_predictions.csv"), "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["true_idx", "pred_idx", "true_label", "pred_label", "p_pred"])
        for i, (t, p) in enumerate(zip(y_true, y_pred)):
            w.writerow([int(t), int(p), LEVELS[int(t)], LEVELS[int(p)],
                        float(probs[i, int(p)])])

    del trainer, model
    gc.collect()
    torch.cuda.empty_cache()
    print("saved", c.output_dir, "\n")

## Run

`RUN_TEST_EVAL` is the same gate used in the other notebooks. Flip it once.
Existing `test_metrics.json` files are skipped unless `overwrite=True`, so
re-running the cell after a crash resumes rather than repeating.

In [6]:
RUN_TEST_EVAL = False   # <-- set to True only when you mean it

if RUN_TEST_EVAL:
    for c in CONDITIONS:
        run_test(c)
else:
    print("RUN_TEST_EVAL is False -- test set untouched.")

=== T5Gemma CORN | runs/t5gemma2_corn\checkpoint-1900 ===
data/test.jsonl: kept 604, skipped 0 out-of-scope


Loading weights:   0%|          | 0/1327 [00:00<?, ?it/s]

missing 0 | unexpected 0


logits (604, 7)  rows sum to 1: True
test: {
  "test_loss": 0.9529,
  "test_model_preparation_time": 0.004,
  "test_qwk": 0.88,
  "test_mae": 0.5166,
  "test_accuracy": 0.5265,
  "test_adjacent_accuracy": 0.9619,
  "test_precision_macro": 0.4875,
  "test_recall_macro": 0.4092,
  "test_f1_macro": 0.4136,
  "test_precision_weighted": 0.5381,
  "test_recall_weighted": 0.5265,
  "test_f1_weighted": 0.515,
  "test_runtime": 9.1033,
  "test_samples_per_second": 66.35,
  "test_steps_per_second": 8.349
}
saved runs/t5gemma2_corn 

=== T5Gemma nominal | runs/t5gemma2_nominal\checkpoint-1900 ===
data/test.jsonl: kept 604, skipped 0 out-of-scope


Loading weights:   0%|          | 0/1327 [00:00<?, ?it/s]

missing 0 | unexpected 0


... (6 outputs left out)

Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

[transformers] ModernBertModel LOAD REPORT from: answerdotai/ModernBERT-large
Key               | Status     |  | 
------------------+------------+--+-
head.norm.weight  | UNEXPECTED |  | 
head.dense.weight | UNEXPECTED |  | 
decoder.bias      | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


missing 0 | unexpected 0


logits (604, 8)  rows sum to 1: True
test: {
  "test_loss": 1.1675,
  "test_model_preparation_time": 0.002,
  "test_qwk": 0.852,
  "test_mae": 0.5728,
  "test_accuracy": 0.5116,
  "test_adjacent_accuracy": 0.9272,
  "test_precision_macro": 0.3721,
  "test_recall_macro": 0.3734,
  "test_f1_macro": 0.3593,
  "test_precision_weighted": 0.4753,
  "test_recall_weighted": 0.5116,
  "test_f1_weighted": 0.4807,
  "test_runtime": 1.6981,
  "test_samples_per_second": 355.684,
  "test_steps_per_second": 44.755
}
saved runs/modernbert_nominal 



## Test results table

In [7]:
def load_preds(run_dir, split="test"):
    d = pd.read_csv(f"{run_dir}/{split}_predictions.csv")
    d = d.rename(columns={"true_idx": "gold", "pred_idx": "pred",
                          "true_label": "gold_label", "pred_label": "pred_label"})
    d["correct"] = (d["gold"] == d["pred"]).astype(int)
    d["adjacent"] = (d["gold"] - d["pred"]).abs().le(1).astype(int)
    return d


def ece(d, n_bins=10):
    b = pd.cut(d["p_pred"], np.linspace(0, 1, n_bins + 1))
    g = d.groupby(b, observed=True)
    return float(((g["correct"].mean() - g["p_pred"].mean()).abs() * g.size()).sum() / len(d))


ORDER = ["qwk", "mae", "exact_acc", "adjacent_acc", "f1_macro"]
RENAME = {"accuracy": "exact_acc", "adjacent_accuracy": "adjacent_acc"}

cols = {}
for c in CONDITIONS:
    path = f"{c.output_dir}/test_predictions.csv"
    if not os.path.exists(path):
        continue
    d = load_preds(c.output_dir)
    m = evaluate_predictions(d["gold"].to_numpy(), d["pred"].to_numpy())
    m = {RENAME.get(k, k): v for k, v in m.items()}
    m["ece"] = ece(d)
    m["mean_conf"] = d["p_pred"].mean()
    m["n"] = len(d)
    cols[c.name] = m

if not cols:
    raise SystemExit("No test_predictions.csv found -- run the RUN_TEST_EVAL cell first.")

table = pd.DataFrame(cols)
table = table.loc[[k for k in ORDER + ["ece", "mean_conf", "n"] if k in table.index]]
table = table.round(3)
table.to_csv("runs/test_summary.csv")
table

,T5Gemma CORN,T5Gemma nominal,ModernBERT CORN,ModernBERT nominal
qwk,0.880,0.882,0.849,0.852
mae,0.517,0.523,0.609,0.573
exact_acc,0.526,0.513,0.467,0.512
adjacent_acc,0.962,0.964,0.932,0.927
f1_macro,0.414,0.377,0.320,0.359
ece,0.391,0.427,0.030,0.030
mean_conf,0.918,0.933,0.468,0.485
n,604.000,604.000,604.000,604.000


## Confusion matrices (test)

In [8]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.metrics import confusion_matrix

panels = [c for c in CONDITIONS
          if os.path.exists(f"{c.output_dir}/test_predictions.csv")]
if not panels:
    raise SystemExit("No test_predictions.csv found -- run the RUN_TEST_EVAL cell first.")
n = len(LEVELS)


def counts_norm(run_dir):
    d = load_preds(run_dir)
    counts = confusion_matrix(d["gold"], d["pred"], labels=list(range(n)))
    row_sums = counts.sum(axis=1, keepdims=True)
    norm = np.divide(counts, row_sums, out=np.zeros(counts.shape, dtype=float),
                     where=row_sums != 0)
    return counts, norm


def annot(counts, norm):
    a = np.empty_like(counts, dtype=object)
    for i in range(n):
        for j in range(n):
            a[i, j] = f"{norm[i, j] * 100:.1f}%<br>({counts[i, j]})"
    return a


fig = make_subplots(rows=1, cols=len(panels),
                    subplot_titles=[c.name for c in panels],
                    horizontal_spacing=0.08)
fig.update_layout(coloraxis=dict(colorscale="Greys", cmin=0, cmax=1,
                                 colorbar=dict(title="Row-normalized")))

for k, c in enumerate(panels):
    counts, norm = counts_norm(c.output_dir)
    fig.add_trace(go.Heatmap(z=norm, x=LEVELS, y=LEVELS,
                             text=annot(counts, norm), texttemplate="%{text}",
                             hoverongaps=False, coloraxis="coloraxis"),
                  row=1, col=k + 1)
    fig.update_xaxes(title_text="Predicted", row=1, col=k + 1)
    fig.update_yaxes(title_text="True", autorange="reversed", row=1, col=k + 1)

fig.update_layout(title=dict(text="Confusion matrices (test)"),
                  font=dict(family="Arial", size=13, color="black"),
                  width=480 * len(panels), height=520,
                  margin=dict(l=70, r=100, t=110, b=70))
fig.show()

# Styled for poster for ModernBERT Nominal

In [ ]:
import numpy as np, pandas as pd, plotly.graph_objects as go
from sklearn.metrics import confusion_matrix

LEVELS = ["A2","A2+","B1","B1+","B2","B2+","C1","C1+"]; N = len(LEVELS)
RED, NAVY, CARD = "#8E1728", "#22305F", "#FBF9F7"
SCALE = [[0.0, CARD], [0.35, "#E8C4C8"], [0.7, "#C07680"], [1.0, RED]]

d = pd.read_csv("runs/modernbert_nominal/test_predictions.csv")
gold = d["true_idx"] if "true_idx" in d else d["gold"]
pred = d["pred_idx"] if "pred_idx" in d else d["pred"]

counts = confusion_matrix(gold, pred, labels=list(range(N)))
rs = counts.sum(1, keepdims=True)
norm = np.divide(counts, rs, out=np.zeros(counts.shape, float), where=rs != 0)

fig = go.Figure(go.Heatmap(z=norm, x=LEVELS, y=LEVELS, colorscale=SCALE,
                           zmin=0, zmax=1, xgap=3, ygap=3,
                           showscale=False, hoverinfo="skip"))
for i in range(N):
    for j in range(N):
        if counts[i, j]:
            fig.add_annotation(x=LEVELS[j], y=LEVELS[i], showarrow=False,
                text=f"<b>{norm[i,j]*100:.0f}%</b><br>({counts[i,j]})",
                font=dict(family="Poppins", size=20,
                          color=CARD if norm[i,j] > 0.55 else NAVY))

fig.update_layout(
    xaxis=dict(title="Predicted", showgrid=False, tickfont=dict(size=17)),
    yaxis=dict(title="True", autorange="reversed", showgrid=False,
               tickfont=dict(size=17), scaleanchor="x"),
    font=dict(family="Times New Roman", size=18, color=NAVY),
    paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)",
    width=760, height=740, margin=dict(l=80, r=20, t=20, b=70))
fig.show()

# fig.write_image("confusion_matrix.svg")
# fig.write_image("confusion_matrix.png", scale=4)

# Calibration 
### first check that the CORN and nominal probabilities are comparable

In [21]:
c = CONDITIONS[0]                    # T5Gemma CORN
tokenizer = AutoTokenizer.from_pretrained(c.model_id)
test_ds = CEFRDataset(TEST_PATH, tokenizer, c.max_length)
model = build_model(c)
model.load_state_dict(load_file(os.path.join(resolve_ckpt(c), "model.safetensors")),
                      strict=False)
args = TrainingArguments(output_dir="tmp_eval", per_device_eval_batch_size=c.batch_size,
                         bf16=c.bf16, report_to=[], do_train=False)
trainer = OrdinalTrainer(model=model, args=args, mode=c.mode,
                         processing_class=tokenizer,
                         data_collator=DataCollatorWithPadding(tokenizer))
logits_t = as_logits(trainer.predict(test_ds).predictions)

p = corn_probas(logits_t).numpy()
thresh = corn_label_from_logits(logits_t).numpy()
print("disagreement:", (p.argmax(1) != thresh).mean())

data/test.jsonl: kept 604, skipped 0 out-of-scope


Loading weights:   0%|          | 0/1327 [00:00<?, ?it/s]

disagreement: 0.0


In [23]:
BINS = [0, .3, .4, .5, .6, .7, 1.0]
for name, run_dir in [("T5Gemma CORN", "runs/t5gemma2_corn"),
                      ("ModernBERT nominal", "runs/modernbert_nominal")]:
    d = load_preds(run_dir, split="test")
    d["adjacent"] = (d["gold"] - d["pred"]).abs().le(1).astype(int)
    b = pd.cut(d["p_pred"], BINS)
    print(f"\n{name}")
    print(d.groupby(b, observed=True).agg(
        n=("correct", "size"), exact=("correct", "mean"),
        adjacent=("adjacent", "mean"), mean_conf=("p_pred", "mean")).round(3))


T5Gemma CORN
              n  exact  adjacent  mean_conf
p_pred                                     
(0.5, 0.6]   31  0.516     0.968      0.552
(0.6, 0.7]   36  0.528     0.944      0.651
(0.7, 1.0]  537  0.527     0.963      0.957

ModernBERT nominal
              n  exact  adjacent  mean_conf
p_pred                                     
(0.0, 0.3]    6  0.500     0.667      0.282
(0.3, 0.4]   98  0.398     0.878      0.365
(0.4, 0.5]  215  0.465     0.902      0.451
(0.5, 0.6]  244  0.582     0.967      0.543
(0.6, 0.7]   39  0.615     1.000      0.635
(0.7, 1.0]    2  0.500     0.500      0.720
